# 02 — LLM feature discovery and extraction with `llm-feature-gen`

**Goal.** Show the two-step pipeline the project already uses, and make it reproducible.

The library works in two stages:

1. **Discovery** — show the model a pile of transcripts *without labels* and ask it to propose
   a feature schema (feature name, description, possible values). Written to a JSON file.
2. **Generation** — for every document, ask the model to assign a value to each feature in
   that schema. Written to one CSV per class.

**Which data goes where.** Discovery runs on `overview/` (= `train.extra`), which has no
labels at all. That is deliberate: the schema must be designed without ever seeing the class
structure. Generation then runs on `train/`. The test folder is not used.

> **Cost note.** A full extraction is one LLM call per document. This notebook therefore
> defaults to `RUN_LLM = False` and loads the feature table that was already produced
> (`OutputsQwen/`). Set `RUN_LLM = True` when you have the endpoint available and want to
> regenerate. The code that runs is the same either way.

In [5]:
!pip install llm-feature-gen

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.9/44.9 kB 1.2 MB/s eta 0:00:00


In [6]:
import os
from dotenv import load_dotenv
load_dotenv()  # loads LOCAL_OPENAI_API_KEY from the project-root .env (never hardcode the key)

import json
from pathlib import Path

import pandas as pd

# ---------------------------------------------------------------- configuration
RUN_LLM = True

BASE_URL = "llmlite.vse.cz"
API_KEY = os.environ["LOCAL_OPENAI_API_KEY"]
MODEL = "qwen3.6:122b"

DATA = Path("fileDataset")
if not (DATA / "train").exists() and (DATA / "fileDataset" / "train").exists():
    DATA = DATA / "fileDataset"

OUT = Path("outputs")
OUT.mkdir(exist_ok=True)
print("RUN_LLM =", RUN_LLM)

RUN_LLM = True


## The provider

`LocalProvider` is the right class for an OpenAI-compatible endpoint such as the school's
Ollama server. (`OpenAIProvider` does not accept a custom `base_url` on its non-Azure path,
so it will not work against `llm.vse.cz`.)

In [7]:
from llm_feature_gen.providers.local_provider import LocalProvider

if RUN_LLM:
    provider = LocalProvider(
        base_url=BASE_URL,
        api_key=API_KEY,
        default_text_model=MODEL,
        temperature=0.0,              # deterministic-ish: we want repeatable features
    )
    print("provider ready:", provider.base_url, "|", provider.text_model)
else:
    provider = None
    print("provider not created (RUN_LLM is False)")

provider ready: llmlite.vse.cz | qwen3.6:122b


## Step 1 — Discovery prompt

The library ships a default discovery prompt, but it is written for a *two-hidden-class*
setting and says so explicitly. That is a problem here: telling the model there are two groups
invites it to write feature descriptions of the form *"one group does X, the other does Y"*,
which is exactly what happened in the first run of this project. The schema then encodes the
class structure before any labels are used.

The prompt below is label-blind: it never mentions groups, classes, diagnoses or health.

In [8]:
DISCOVERY_PROMPT = """\
You are an expert in computational linguistics and explainable machine learning.

Your task is to discover candidate, human-interpretable linguistic features from Czech
spontaneous picture-description transcripts.

IMPORTANT
- The transcripts are transcriptions of spontaneous speech produced while participants
  describe the same picture.
- You are NOT given any labels, groups, or categories.
- Do NOT infer or name diagnoses, cognitive impairment, health status, intelligence, or
  speaker ability.
- Do NOT define a feature by reference to any class or group of speakers.
- Do NOT use demographic characteristics.
- Features must be observable from the transcript itself.

Consider linguistic and discourse dimensions such as: lexical diversity and repetition,
lexical specificity, entity coverage, relations between objects, spatial organization,
syntactic complexity, discourse coherence, referential specificity, pronoun use, semantic
completeness, perseveration-like repetition, self-correction, hesitation and disfluency,
epistemic uncertainty, verb usage, information density, and level of descriptive detail.
Use the actual transcripts to decide which dimensions are appropriate.

A good feature must satisfy ALL of the following:
1. It can be observed from a single transcript.
2. It has a clear operational definition.
3. A human researcher could reliably assign its value.
4. It represents a distinct linguistic dimension.
5. Its values are mutually exclusive and reasonably exhaustive.
6. It applies to spontaneous picture descriptions.
7. It does not encode a diagnostic judgement.

Generate 15-20 candidate features. Avoid redundant features that measure the same underlying
phenomenon. Avoid vague value labels such as "good", "bad", "normal", or "impaired".

Output ONLY valid JSON:
{
  "proposed_features": [
    {"feature": "feature_name",
     "description": "operational definition",
     "possible_values": ["value1", "value2", "value3"]}
  ]
}
"""
print(f"prompt length: {len(DISCOVERY_PROMPT)} chars")
for word in ["group", "class", "diagnos", "impair", "patient"]:
    print(f"  contains '{word}': {word in DISCOVERY_PROMPT.lower()}")

prompt length: 1970 chars
  contains 'group': True
  contains 'class': True
  contains 'diagnos': True
  contains 'impair': True
  contains 'patient': False


## Step 1 (run) — discover a schema from the unlabelled pool

In [9]:
from llm_feature_gen.discover import discover_features_from_texts

SCHEMA_PATH = OUT / "discovered_text_features_v1.json"

if RUN_LLM:
    discovered = discover_features_from_texts(
        texts_or_file=str(DATA / "overview"),   # unlabelled pool only
        prompt=DISCOVERY_PROMPT,
        provider=provider,
        as_set=True,                            # one request over all texts -> one shared schema
        output_dir=str(OUT),
        output_filename=SCHEMA_PATH.name,
    )
    print("discovered schema written to", SCHEMA_PATH)
else:
    print("skipped — using the schema already produced in OutputsQwen/")

FileNotFoundError: Path not found: fileDataset/overview

## Step 2 — Generate feature values for the training documents

`generate_features_from_texts` expects a root folder containing one subfolder per class. Our
`train/` folder already has `negative/` and `positive/`, so it works directly. `Class` in the
output CSV is just the folder name — it is a bookkeeping column, not something the model saw.

In [ ]:
from llm_feature_gen.generate import generate_features_from_texts

if RUN_LLM:
    paths = generate_features_from_texts(
        root_folder=str(DATA / "train"),
        discovered_features_path=str(SCHEMA_PATH),
        output_dir=str(OUT / "train_features_v1"),
        classes=["negative", "positive"],
        merge_to_single_csv=True,
        merged_csv_name="train_all_feature_values.csv",
        provider=provider,
    )
    print(paths)
else:
    print("skipped — using OutputsQwen/train_all_feature_values.csv")

skipped — using OutputsQwen/train_all_feature_values.csv


## Load the feature table

Either the one we just produced, or the existing one from `OutputsQwen/`.

In [ ]:
CANDIDATES = [
    OUT / "train_features_v1" / "train_all_feature_values.csv",
    Path("OutputsQwen") / "train_all_feature_values.csv",
    Path("train_all_feature_values.csv"),
]
feat_path = next((p for p in CANDIDATES if p.exists()), None)
if feat_path is None:
    raise FileNotFoundError(f"no feature CSV found; looked in {[str(p) for p in CANDIDATES]}")

features = pd.read_csv(feat_path)
FEATURE_COLS = [c for c in features.columns if c not in ("File", "Class", "raw_llm_output")]

print("loaded:", feat_path)
print(f"{len(features)} rows x {len(FEATURE_COLS)} features")
print("classes:", features.Class.value_counts().to_dict())
print("\nfeatures:")
for c in FEATURE_COLS:
    print("  -", c)

FileNotFoundError: no feature CSV found; looked in ['outputs/train_features_v1/train_all_feature_values.csv', 'OutputsQwen/train_all_feature_values.csv', 'train_all_feature_values.csv']

## What one row looks like

Each feature is a short categorical label. `raw_llm_output` keeps the model's original JSON,
which is useful when a value looks wrong and you want to see what was actually returned.

In [ ]:
row = features.iloc[0]
print("file:", row.File, "| class:", row.Class, "\n")
for c in FEATURE_COLS:
    print(f"  {c:38} {row[c]}")

## Coverage check — did every document get every feature?

In [ ]:
missing = features[FEATURE_COLS].isna().sum()
print("missing values per feature:")
print(missing.to_string() if missing.sum() else "  none — every document has every feature")

levels = {c: sorted(features[c].dropna().unique()) for c in FEATURE_COLS}
print("\ndistinct values actually used:")
for c, v in levels.items():
    print(f"  {c:38} {len(v)}  {v}")

In [ ]:
features.to_csv(OUT / "features_v1_train.csv", index=False)
json.dump({"feature_columns": FEATURE_COLS}, open(OUT / "feature_columns.json", "w"), indent=2)
print(f"wrote outputs/features_v1_train.csv")

### Summary

* Discovery runs on the unlabelled `overview/` pool; generation runs on `train/`.
* The discovery prompt was rewritten to be label-blind. The original library default tells the
  model there are two hidden classes, and the first version of this project's schema contained
  descriptions of the form *"one group tends to… while the other…"*. That is worth mentioning
  to your supervisor — it is a methodological fix, not a cosmetic one.
* The result is 10 categorical features over 241 training documents.
* Whether those features are any good is notebook 03.